# 8 - Generating Training Data

The composed scenarios feed two SFT emitters: label-preserving classifier records, and grounded generative pairs validated against a golden contract. No store is needed when bases are user-supplied.

In [ ]:
import sys, pathlib
import proofloop.suite as gt
cat = gt.Catalog.load()
print(cat.summary())

In [ ]:
items = gt.UserBaseSource([
    {'query': 'I was overcharged on my overdraft', 'answer': 'complaint'},
    {'query': 'How do I close my account?',        'answer': 'inquiry'},
]).items()
suite = gt.resolve(cat, ['exact_recall'], ['clarity', 'noise'], mode='cross')
scns = gt.compose(suite, items, n_runs=4, seed=1)
recs = gt.emit_classifier_sft(scns)
print('classifier records:', len(recs), '| labels:', {r['label'] for r in recs})

Draft pairs are validated against a golden contract (cite the keyword, hold the byte-exact number, avoid forbidden phrases); contract-failing pairs are dropped.

In [ ]:
goldens = {'overdraft': {'citation_keywords': ['overdraft'],
                         'required_numbers': ['35'],
                         'forbidden_phrases': ['we will waive']}}
pol = [gt.QAItem(qid='p0', query='charged $35 overdraft', answer=None,
                 metadata={'policy_id': 'overdraft'})]
psc = gt.compose(gt.resolve(cat, ['exact_recall'], ['clarity'], mode='cross'), pol, n_runs=2)
good = lambda s: 'Your overdraft fee of $35 may be reversed once per year.'
bad  = lambda s: 'We will waive your fee.'
kept_g, drop_g = gt.emit_draft_sft(psc, good, goldens=goldens)
kept_b, drop_b = gt.emit_draft_sft(psc, bad,  goldens=goldens)
print('good kept/dropped:', len(kept_g), len(drop_g))
print('bad  kept/dropped:', len(kept_b), len(drop_b))

## Self-check

In [ ]:
assert {r['label'] for r in recs} == {'complaint', 'inquiry'}   # labels preserved
assert len(kept_g) == 2 and len(kept_b) == 0                    # contract enforced
print('OK - generating training data')